In [1]:
"""
Plant Disease Detection - Custom CNN + 5-Fold Cross Validation
"""

import os
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt
import random
from datetime import datetime

# ------------------------- CONFIG -------------------------
DATASET_DIR = r"C:\Users\natra\Downloads\dl project dataset\PlantVillage" 
IMAGE_SIZE = (128, 128)  
BATCH_SIZE = 32
EPOCHS = 15
N_SPLITS = 5
MAX_IMAGES_PER_CLASS = 400
OUTPUT_DIR = "cnn_kfold_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------- LOAD DATA -------------------------
print("📦 Loading dataset...")

X = []
y = []
classes = sorted([d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d))])

for label in classes:
    folder_path = os.path.join(DATASET_DIR, label)
    files = os.listdir(folder_path)
    random.shuffle(files)
    files = files[:MAX_IMAGES_PER_CLASS]

    for file in files:
        img_path = os.path.join(folder_path, file)
        img = cv2.imread(img_path)
        if img is None:
            continue
        img = cv2.resize(img, IMAGE_SIZE)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        X.append(img)
        y.append(label)

X = np.array(X, dtype=np.float32) / 255.0
y = np.array(y)

print(f"✅ Loaded {len(X)} images from {len(classes)} classes (~{MAX_IMAGES_PER_CLASS} per class max).")

le = LabelEncoder()
y_encoded = le.fit_transform(y)
num_classes = len(classes)
print(f"Number of classes: {num_classes}")

# ------------------------- MODEL FUNCTION -------------------------
def build_cnn_model(input_shape=(128,128,3), num_classes=2, dropout_rate=0.4):
    model = Sequential([
        Conv2D(32, (3,3), activation='relu', padding='same', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling2D(2,2),

        Conv2D(64, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(2,2),

        Conv2D(128, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(2,2),

        Conv2D(256, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(2,2),

        Flatten(),
        Dense(512, activation='relu'),
        Dropout(dropout_rate),
        Dense(num_classes, activation='softmax')
    ])
    return model

# ------------------------- DATA AUGMENTATION -------------------------
datagen = ImageDataGenerator(
    rotation_range=25,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='reflect'
)

# ------------------------- 5-FOLD CROSS VALIDATION -------------------------
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
fold_accuracies = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_encoded), 1):
    print(f"\n================ Fold {fold}/{N_SPLITS} ================")

    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y_encoded[train_idx], y_encoded[val_idx]

    # Build model
    model = build_cnn_model(input_shape=(*IMAGE_SIZE,3), num_classes=num_classes, dropout_rate=0.4)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

    # Callbacks
    fold_time = datetime.now().strftime('%Y%m%d_%H%M%S')
    model_path = os.path.join(OUTPUT_DIR, f"cnn_fold{fold}_{fold_time}.keras")
    callbacks = [
        ModelCheckpoint(model_path, monitor='val_accuracy', save_best_only=True, verbose=1),
        EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
    ]

    # Train
    history = model.fit(
        datagen.flow(X_train, y_train, batch_size=BATCH_SIZE),
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1
    )

    # Evaluate
    val_pred = np.argmax(model.predict(X_val), axis=1)
    acc = accuracy_score(y_val, val_pred)
    fold_accuracies.append(acc)
    print(f"✅ Fold {fold} Accuracy: {acc:.4f}")
    print(classification_report(y_val, val_pred, target_names=le.classes_))

    # Save accuracy/loss plots
    plt.figure(figsize=(8,4))
    plt.plot(history.history['accuracy'], label='train_acc')
    plt.plot(history.history['val_accuracy'], label='val_acc')
    plt.title(f'Fold {fold} Accuracy')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(OUTPUT_DIR, f'fold{fold}_accuracy.png'))
    plt.close()

    plt.figure(figsize=(8,4))
    plt.plot(history.history['loss'], label='train_loss')
    plt.plot(history.history['val_loss'], label='val_loss')
    plt.title(f'Fold {fold} Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(OUTPUT_DIR, f'fold{fold}_loss.png'))
    plt.close()

# ------------------------- FINAL SUMMARY -------------------------
print("\n====================== FINAL RESULTS ======================")
print(f"Average Accuracy over {N_SPLITS} folds: {np.mean(fold_accuracies):.4f}")
print(f"Standard Deviation: {np.std(fold_accuracies):.4f}")

summary_path = os.path.join(OUTPUT_DIR, "kfold_summary.txt")
with open(summary_path, "w") as f:
    for i, acc in enumerate(fold_accuracies, 1):
        f.write(f"Fold {i}: {acc:.4f}\n")
    f.write(f"\nAverage Accuracy: {np.mean(fold_accuracies):.4f}\n")
    f.write(f"Std Dev: {np.std(fold_accuracies):.4f}\n")

print(f"\n📁 All results saved to: {OUTPUT_DIR}")


📦 Loading dataset...
✅ Loaded 4125 images from 11 classes (~400 per class max).
Number of classes: 11

================ Fold 1/5 ================


C:\Users\natra\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/15


C:\Users\natra\anaconda3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 262ms/step - accuracy: 0.5437 - loss: 1.8153
Epoch 1: val_accuracy improved from -inf to 0.09697, saving model to cnn_kfold_outputs\cnn_fold1_20251014_213402.keras
104/104 ━━━━━━━━━━━━━━━━━━━━ 33s 283ms/step - accuracy: 0.5450 - loss: 1.8086 - val_accuracy: 0.0970 - val_loss: 5.0150 - learning_rate: 1.0000e-04
Epoch 2/15
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 261ms/step - accuracy: 0.8450 - loss: 0.4593
Epoch 2: val_accuracy improved from 0.09697 to 0.09818, saving model to cnn_kfold_outputs\cnn_fold1_20251014_213402.keras
104/104 ━━━━━━━━━━━━━━━━━━━━ 30s 278ms/step - accuracy: 0.8450 - loss: 0.4594 - val_accuracy: 0.0982 - val_loss: 7.3494 - learning_rate: 1.0000e-04
Epoch 3/15
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 261ms/step - accuracy: 0.8846 - loss: 0.3442
Epoch 3: val_accuracy improved from 0.09818 to 0.17939, saving model to cnn_kfold_outputs\cnn_fold1_20251014_213402.keras
104/104 ━━━━━━━━━━━━━━━━━━━━ 30s 279ms/step - accuracy: 0.8846 - loss: 0.3441 - val_accu

...

104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 263ms/step - accuracy: 0.5914 - loss: 1.7182
Epoch 1: val_accuracy improved from -inf to 0.09697, saving model to cnn_kfold_outputs\cnn_fold4_20251014_215537.keras
104/104 ━━━━━━━━━━━━━━━━━━━━ 34s 284ms/step - accuracy: 0.5927 - loss: 1.7115 - val_accuracy: 0.0970 - val_loss: 5.7742 - learning_rate: 1.0000e-04
Epoch 2/15
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 262ms/step - accuracy: 0.8560 - loss: 0.4234
Epoch 2: val_accuracy improved from 0.09697 to 0.20606, saving model to cnn_kfold_outputs\cnn_fold4_20251014_215537.keras
104/104 ━━━━━━━━━━━━━━━━━━━━ 30s 280ms/step - accuracy: 0.8560 - loss: 0.4234 - val_accuracy: 0.2061 - val_loss: 9.4703 - learning_rate: 1.0000e-04
Epoch 3/15
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 262ms/step - accuracy: 0.8936 - loss: 0.2985
Epoch 3: val_accuracy improved from 0.20606 to 0.24727, saving model to cnn_kfold_outputs\cnn_fold4_20251014_215537.keras
104/104 ━━━━━━━━━━━━━━━━━━━━ 30s 280ms/step - accuracy: 0.8935 - loss: 0.2986 - val_accu

C:\Users\natra\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/15


C:\Users\natra\anaconda3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 262ms/step - accuracy: 0.5736 - loss: 1.7180
Epoch 1: val_accuracy improved from -inf to 0.03879, saving model to cnn_kfold_outputs\cnn_fold5_20251014_220310.keras
104/104 ━━━━━━━━━━━━━━━━━━━━ 33s 284ms/step - accuracy: 0.5747 - loss: 1.7120 - val_accuracy: 0.0388 - val_loss: 5.5302 - learning_rate: 1.0000e-04
Epoch 2/15
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 263ms/step - accuracy: 0.8398 - loss: 0.4928
Epoch 2: val_accuracy improved from 0.03879 to 0.10788, saving model to cnn_kfold_outputs\cnn_fold5_20251014_220310.keras
104/104 ━━━━━━━━━━━━━━━━━━━━ 30s 281ms/step - accuracy: 0.8399 - loss: 0.4925 - val_accuracy: 0.1079 - val_loss: 6.9042 - learning_rate: 1.0000e-04
Epoch 3/15
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 262ms/step - accuracy: 0.8787 - loss: 0.3467
Epoch 3: val_accuracy improved from 0.10788 to 0.25697, saving model to cnn_kfold_outputs\cnn_fold5_20251014_220310.keras
104/104 ━━━━━━━━━━━━━━━━━━━━ 30s 280ms/step - accuracy: 0.8788 - loss: 0.3465 - val_accu